# ML Capstone: ทำนายการซื้อซ้ำของลูกค้า

Notebook เริ่มต้นสำหรับโปรเจกต์ ML capstone จาก Data Career Lab — ไม่จำเป็นต้องเริ่มจากศูนย์

อ่าน rubric ในบทเรียน `ML Capstone: โปรเจกต์ end-to-end` บนเว็บก่อน แล้วเติมโค้ดและคำตอบในเซลล์ที่มี `TODO`
โครงสร้าง: baseline → แบ่งข้อมูล → pipeline → เทียบโมเดลด้วย CV → จูน → อธิบาย → ทดสอบครั้งสุดท้าย → สรุป

In [ ]:
import os
if not os.path.exists('data-career-lab'):
    os.system('git clone https://github.com/Phakinza007/data-career-lab.git')
os.chdir('data-career-lab')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

df = pd.read_csv('public/data/ml_customers.csv')
df.head()

## 1. Baseline

อัตราซื้อซ้ำเท่าไร? โมเดลที่เดาคลาสส่วนใหญ่เสมอได้ accuracy เท่าไร? (ทุกโมเดลต้องเอาชนะค่านี้)

In [ ]:
y = df['repeated']
print('repeat rate:', round(y.mean(), 4))
print('baseline accuracy:', round(y.value_counts(normalize=True).max(), 4))

## 2. แบ่งข้อมูล (กัน test ไว้ก่อน)

ตรวจว่า feature ทุกตัวรู้ได้ ณ เวลาที่ต้องทำนาย — แล้วแตะ `X_test` แค่ครั้งเดียวตอนท้าย

In [ ]:
X = pd.get_dummies(df.drop(columns=['customer_id', 'repeated', 'first_order_value']), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
X_train.shape, X_test.shape

## 3. เทียบโมเดลด้วย cross-validation (เฉพาะ train)

รายงาน mean ± std ของ AUC และ accuracy ของอย่างน้อย 3 โมเดล เช่น logistic regression, decision tree / random forest, gradient boosting

In [ ]:
models = {
    'baseline': DummyClassifier(),
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    # TODO: เพิ่มโมเดลอื่น เช่น DecisionTreeClassifier(max_depth=3, random_state=42),
    #       RandomForestClassifier(n_estimators=100, min_samples_leaf=10, random_state=42),
    #       GradientBoostingClassifier(random_state=42)
}
rows = []
for name, model in models.items():
    auc = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc')
    acc = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
    rows.append({'model': name, 'auc_mean': auc.mean(), 'auc_std': auc.std(), 'acc_mean': acc.mean()})
pd.DataFrame(rows).round(4)

## 4. จูน hyperparameter (ใช้ train เท่านั้น)

In [ ]:
# TODO: GridSearchCV บนโมเดลที่คุณเลือก (grid เล็กๆ) แล้วดู best_params_ / best_score_

## 5. อธิบายโมเดล

feature ไหนที่โมเดลพึ่ง? ใช้ permutation importance บนชุด test (หรือ coefficient ของ logistic regression)

In [ ]:
# TODO: permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=42, scoring='roc_auc')

## 6. ทดสอบครั้งสุดท้าย (ครั้งเดียว)

In [ ]:
# TODO: fit โมเดลที่เลือกบน train ทั้งหมด แล้วรายงาน AUC และ accuracy บน X_test เทียบกับ baseline

## 7. สรุป

เขียน 3–5 บรรทัด: โมเดลที่เลือกและเหตุผล, ดีกว่า baseline แค่ไหน (พร้อมตัวเลข), feature สำคัญ, ข้อจำกัดของผล (ข้อมูลน้อย สัญญาณอ่อน ความแกว่งข้าม seed/fold) และสิ่งที่จะทำต่อ

In [ ]:
# เขียนสรุปเป็นคอมเมนต์ตรงนี้